In [34]:
import spacy
import nltk
import pandas as pd
import pickle
from rapidfuzz import fuzz
import shutil

# 0) Import data

In [31]:
# Import metadata
df = pd.read_csv("../data/document_data_clean_filtered.csv")

In [35]:
#Create_backup:
shutil.copy("../data/document_data_clean_filtered.csv", "../data/document_data_clean_filtered_auto_backup.csv")
# Save df:
df.to_csv("../data/document_data_clean_filtered.csv", index=False)

# 1) Occurrences

In [26]:
def get_occurrences(ark, target="eucalyptus", ratio=75):
    with open(f"../data/corpus_txt/{ark}.txt", "r", encoding='utf-8') as f:
        text = f.read().split()

    if type(target) is not list:
        occurrences = 0
        for word in text:
            if target == "eucalyptus":
                if fuzz.ratio("eucalyptus", word.lower()) > ratio or "eucalypt" in word.lower() or "encalypt" in word.lower() or "eucalipt" in word.lower():
                    occurrences += 1
            else:
                if fuzz.ratio(target, word.lower()) > ratio or target in word.lower():
                    occurrences += 1
        
        return occurrences

    else:
        occurrences_list = [0 for x in target]
        for word in text:
            for i, word_target in enumerate(target):
                if word_target == "eucalyptus":
                    if fuzz.ratio("eucalyptus", word.lower()) > ratio or "eucalypt" in word.lower() or "encalypt" in word.lower() or "eucalipt" in word.lower():
                        occurrences_list[i] += 1
                else:
                    if fuzz.ratio(word_target, word.lower()) > ratio or word_target in word.lower():
                        occurrences_list[i] += 1
    
        return occurrences_list

In [28]:
docs_occurrences = []
for doc in df["ark"].to_list():
    docs_occurrences.append(get_occurrences(doc))

#df["occurences"] = docs_occurrences

In [32]:
# Here it is separated in order to accomodate work on df in another notebook. Once the operation on topics is finished
# we can then import df again and run this cell 
df["occurences"] = docs_occurrences

# 2) Apply spacy to all texts

In [20]:
# Load the pre-trained model
nlp = spacy.load("fr_core_news_lg")

In [21]:
docs_ner = []

for ark in df["ark"].to_list():
    with open(f"../data/corpus_txt/{ark}.txt", "r", encoding='utf-8') as f:
            text = f.read()
    docs_ner.append(nlp(text))

#Save docs_ner
with open("docs_ner.pkl", "wb") as fp:
    pickle.dump(docs_ner, fp)

ValueError: [E088] Text of length 3676236 exceeds maximum of 1000000. The parser and NER models require roughly 1GB of temporary memory per 100,000 characters in the input. This means long texts may cause memory allocation errors. If you're not using the parser or NER, it's probably safe to increase the `nlp.max_length` limit. The limit is in number of characters, so you can check whether your inputs are too long by checking `len(text)`.